# Protection & Audit (Intervalle)

The core SDC workflow, end to end:

1. **Compute** tables from microdata
2. **Suppress** unsafe cells (OPT / MOD / RND)
3. **Audit** — compute the *realized feasibility intervals* for each
   suppressed cell (the legacy `intervalle.exe`, now in-process)
4. **Export** the protected table (CSV or legacy INTERMEDIATE)

Start with the DataFrame API, then inspect the equivalent lower-level
batch workflow. `protect()` suppresses; audit is an explicit next step.
Build this development branch first; the new API awaits release.

## The audit in one paragraph

A suppressed cell has a range of values it could take while staying
consistent with the published totals — its **realized lower/upper
bounds** (the *feasibility interval*). Protection requires sufficient
uncertainty to meet the configured protection requirements; a narrow
interval can reveal the value. In legacy Tau-Argus the audit ran in a **separate**
`intervalle.exe`; the rewrite ports it into the engine (`TauAuditJj`
in the `csp` submodule), so it runs **in-process** over a temporary
`.JJ` file with nothing external to install.

## Setup

In [1]:
import os
from pathlib import Path
import pytauargus

# Locate the data/ fixtures, which live in the pinned tauargus-engine
# checkout: TAUARGUS_ENGINE_DIR if set, then the sibling tauargus-engine
# checkout of each enclosing directory, then the enclosing directories
# themselves (legacy in-engine layout).
_here = Path.cwd()
_engine_dir = os.environ.get("TAUARGUS_ENGINE_DIR")
_cands = ([_engine_dir] if _engine_dir else []) + [
    str(p / "tauargus-engine") for p in [_here, *_here.parents]
] + [str(p) for p in [_here, *_here.parents]]
DATA = next(
    (Path(c) / "data") for c in _cands
    if (Path(c) / "data" / "tau_testW.asc").exists()
)
OUT = _here / "out"          # scratch dir for notebook outputs (gitignored)
OUT.mkdir(exist_ok=True)

def display_paths(text):
    """Use portable paths in displayed text; engine inputs keep real paths."""
    text = str(text)
    for path, label in ((OUT, "./out"), (DATA, "<repo>/data")):
        text = text.replace(str(path), label).replace(path.as_posix(), label)
    return text

print(f"pytauargus {pytauargus.__version__}")
print("data : <repo>/data")
print("out  : ./out")

pytauargus 0.6.6
data : <repo>/data
out  : ./out


## DataFrame protection, audit and export

Use the same synthetic respondents as the quickstart. OPT chooses
secondary cells so published totals cannot reconstruct primary cells.

In [2]:
import pandas as pd
from pytauargus import protect

# Twelve synthetic respondents; North/Retail has one dominant contributor.
micro = pd.DataFrame({
    "Region": ["North"] * 6 + ["South"] * 6,
    "Sector": ["Retail"] * 3 + ["Services"] * 3
              + ["Retail"] * 3 + ["Services"] * 3,
    "Turnover": [100., 5., 5., 10., 10., 10., 20., 20., 20., 30., 30., 30.],
})
micro

,Region,Sector,Turnover
0,North,Retail,100.0
1,North,Retail,5.0
2,North,Retail,5.0
3,North,Services,10.0
4,North,Services,10.0
5,North,Services,10.0
6,South,Retail,20.0
7,South,Retail,20.0
8,South,Retail,20.0
9,South,Services,30.0


In [3]:
from collections import Counter

protected = protect(
    micro, ["Region", "Sector"], response="Turnover",
    safety_rules="NK(2,75)", suppress="OPT(1)",
    workdir=str(OUT / "02_protect"),
)
table = protected.tables[0]
assert "U" in table.status() and "M" in table.status()
print("status counts:", dict(Counter(table.status())))
table.dataframe()[["Region", "Sector"]].assign(Turnover=table.safe())

status counts: {'S': 5, 'U': 2, 'M': 2}


,Region,Sector,Turnover
0,Total,Total,290.0
1,Total,Retail,170.0
2,Total,Services,120.0
3,North,Total,x
4,North,Retail,x
5,North,Services,30.0
6,South,Total,x
7,South,Retail,x
8,South,Services,90.0


### Audit the returned engine

`safe()` masks the suppressed cells; it does not run an audit.
Call `protected.engine.audit(0)` explicitly (engine table indices
are zero-based). `min`/`max` are the realized feasibility bounds.
The `lower`/`upper` columns in the default result hold protection levels,
and the result is a snapshot: it does not refresh when audit runs.

In [4]:
audit_rows = protected.engine.audit(0)
audit_frame = pd.DataFrame(audit_rows)
assert len(audit_frame) > 0 and not audit_frame["unsafe"].any()
assert ((audit_frame["min"] <= audit_frame["value"])
        & (audit_frame["value"] <= audit_frame["max"])).all()
audit_frame[["cell", "value", "min", "max", "unsafe"]]

,cell,value,min,max,unsafe
0,3,140.0,60.0,170.0,False
1,4,110.0,30.0,140.0,False
2,6,150.0,120.0,230.0,False
3,7,60.0,30.0,140.0,False


### Publish masked values; retain the audit separately

The full DataFrame and `.tab` intermediates contain original values.
Export the selected codes with `safe()` as the response. Keep the
INTERMEDIATE file with realized bounds for internal audit review.

In [5]:
release = table.dataframe()[["Region", "Sector"]].assign(Turnover=table.safe())
release.to_csv(OUT / "02_dataframe_publication.csv", index=False)
protected.engine.write_intermediate_table(
    0, str(OUT / "02_dataframe_audit.tab"), with_audit=True,
)
print("wrote 02_dataframe_publication.csv and 02_dataframe_audit.tab")
release

wrote 02_dataframe_publication.csv and 02_dataframe_audit.tab


,Region,Sector,Turnover
0,Total,Total,290.0
1,Total,Retail,170.0
2,Total,Services,120.0
3,North,Total,x
4,North,Retail,x
5,North,Services,30.0
6,South,Total,x
7,South,Retail,x
8,South,Services,90.0


### Choose another suppression method

Switch to `MOD(1)` for modular suppression. Each call computes its
own tables; use a separate working directory for each result.

In [6]:
modular = protect(
    micro, ["Region", "Sector"], response="Turnover",
    safety_rules="NK(2,75)", suppress="MOD(1)",
    workdir=str(OUT / "02_modular"),
)
print("MOD status counts:", dict(Counter(modular.tables[0].status())))
modular.tables[0].dataframe()[["Region", "Sector"]].assign(
    Turnover=modular.tables[0].safe(),
)

MOD status counts: {'S': 5, 'U': 2, 'M': 2}


,Region,Sector,Turnover
0,Total,Total,290.0
1,Total,Retail,170.0
2,Total,Services,120.0
3,North,Total,x
4,North,Retail,x
5,North,Services,30.0
6,South,Total,x
7,South,Retail,x
8,South,Services,90.0


## 1. Batch API: compute the tables

Run the batch. `NK(n,k)` flags cells whose n largest contributors
exceed k percent of the response — these start as `unsafe_rule`.

In [7]:
from pytauargus.engine import run_batch
from pytauargus.batch import Suppress
from collections import Counter

STATUS_NAMES = {
    1: "safe", 2: "safe_manual", 3: "unsafe_rule", 4: "unsafe_peep",
    5: "unsafe_freq", 6: "unsafe_zero", 7: "unsafe_singleton",
    8: "unsafe_singleton_manual", 9: "unsafe_manual", 10: "protect_manual",
    11: "secondary_unsafe", 12: "secondary_unsafe_manual",
    13: "empty_nonstructural", 14: "empty",
}

eng = run_batch(DATA / "TestRecode.arb")
tau = eng._tau
n = tau.get_total_table_size(0)[1]
before = Counter(tau.get_table_cell_status(0, j)[1] for j in range(n))
print(f"Table 1 — {n} cells, pre-suppression:")
for code_, cnt in sorted(before.items()):
    print(f"  {STATUS_NAMES.get(code_, code_):<24} {cnt}")

Table 1 — 126 cells, pre-suppression:
  safe                     104
  unsafe_rule              1
  empty                    21


## 2. Suppress

Suppression methods available in headless mode:

| Method | Call | Notes |
|--------|------|-------|
| **OPT** | `eng.suppress(Suppress(kind="OPT", tab_no=N))` | Optimal, LP-based. `opt_max_time=0` = no limit (default). |
| **MOD** | `eng.suppress(Suppress(kind="MOD", tab_no=N))` | Modular, heuristic. `mod_max_time=0` (default). |
| **RND** | `eng.suppress(Suppress(kind="RND", tab_no=N, rnd_base=B))` | Controlled rounding; `B` must be ≥ 10 000 000. |

> `opt_max_time`/`mod_max_time` are in **minutes**; `0` means no limit.
> Positive limits are covered by repeated-run regression tests.
> These examples use the default `0` so they can finish without a deadline.
>
> `GH` (global hiding), `NET` (network) and `CTA` need external
> executables and are **not** available headless.

In [8]:
# Suppress table 1 (tab_no is 1-based in the API, as in the .arb grammar)
eng.suppress(Suppress(kind="OPT", tab_no=1))

after = Counter(tau.get_table_cell_status(0, j)[1] for j in range(n))
print("Table 1 — after OPT suppression:")
for code_, cnt in sorted(after.items()):
    print(f"  {STATUS_NAMES.get(code_, code_):<24} {cnt}")

Table 1 — after OPT suppression:
  safe                     101
  unsafe_rule              1
  secondary_unsafe         3
  empty                    21


### RND (controlled rounding) — alternate flow

Controlled rounding needs a base ≥ 10 000 000. The engine computes a
suitable default from the table's cell sizes:

In [9]:
# Suppression mutates state — use a fresh engine for the alternate flow
eng2 = run_batch(DATA / "TestRecode.arb")
base = eng2._min_round_base(0)
print(f"rounding base: {base}")
eng2.suppress(Suppress(kind="RND", tab_no=1, rnd_base=base))
print("RND applied OK")

rounding base: 10000000
RND applied OK


## 3. Audit (Intervalle)

`eng.audit(tab)` (tab is 0-based) computes the **realized
feasibility interval** for every suppressed cell of `tab`. For each
it solves two LPs — minimize the cell, then maximize it, subject to
the published totals and cell bounds — and stores the bounds on the
cell (readable via `get_table_cell` as the last two entries).

Returns a list of dicts, one per suppressed cell:

| Key | Meaning |
|-----|---------|
| `cell` | Cell index (row-major) |
| `min` / `max` | Realized lower / upper bound |
| `value` | Original cell value, withheld from the release table |
| `status` | `u` = primary, `m` = secondary |
| `unsafe` | `True` if the ported audit reports insufficient protection |

In [10]:
rows = eng.audit(0)
print(f"{len(rows)} suppressed cells audited")
print()
for r in rows:
    flag = "   ⚠ UNDER-PROTECTED" if r["unsafe"] else ""
    print(f"  cell {r['cell']:3d}  interval=[{r['min']:>10.1f}, {r['max']:>10.1f}]  "
          f"value={r['value']:>10.1f}  {r['status']}{flag}")

4 suppressed cells audited

  cell  78  interval=[      -0.0,    30118.0]  value=   18150.0  m
  cell  79  interval=[   69831.0,    99949.0]  value=   81799.0  m
  cell  96  interval=[      -0.0,    30118.0]  value=   11968.0  u
  cell  97  interval=[  126854.0,   156972.0]  value=  145004.0  m


### Reading the result

`unsafe=False` means the cell passes the ported audit's protection-level
check. The feasibility interval normally contains the original value;
protection comes from sufficient uncertainty, not from excluding it.

`unsafe=True` means the audit reports insufficient protection. Review
the suppression pattern and protection requirements before publishing.

### Try it yourself

- Suppress table 2 instead: `eng.suppress(Suppress(kind="OPT", tab_no=2))`
  then `eng.audit(1)` — a 12 726-cell table.
- Compare OPT vs MOD on table 1 and diff the status counts.

## Engine errors (0.7.0)

Native engine failures are raised as `TauArgusNativeError` (a
`BatchError` subclass, so existing `except BatchError` handlers
still catch them) carrying:

- `e.code` — the native error code (0 when the call reports none)
- a message with the legacy catalog text for that code

Running OPT on an already-optimised table is rejected by the
native engine with legacy code 5008. `eng` already ran OPT in
section 2, so the second run fails:

In [11]:
from pytauargus import TauArgusNativeError
from pytauargus.batch import BatchError

try:
    eng.suppress(Suppress(kind="OPT", tab_no=1))  # second OPT on `eng`
    raise AssertionError("expected the second OPT to fail")
except TauArgusNativeError as e:
    print(f"{type(e).__name__}: {e}")
    print("code          :", e.code)
    print("is BatchError :", isinstance(e, BatchError))
    assert e.code == 5008

TauArgusNativeError: FullJJ failed (code 5008): Couldn't load JJ-routines
code          : 5008
is BatchError : True


## 4. Export

Two public export paths:

| Format | Call | Notes |
|--------|------|-------|
| CSV (published values) | `eng.write_table(WriteTable(tab_no=1, output_type=1, file=…))` | `output_type=1` is CSV |
| **INTERMEDIATE** (legacy audit file) | `eng.write_intermediate_table(tab, path, with_audit=True)` | One row per cell + realized bounds |

The INTERMEDIATE (WRITETABLE type-5) file is the legacy audit
format: values, statuses, and — with `with_audit=True` (the legacy
`AR+` option) — the realized feasibility bounds and relative-width
percentages for every cell.

In [12]:
from pytauargus.batch import WriteTable

csv_path = str(OUT / "table1.csv")
eng.write_table(WriteTable(tab_no=1, output_type=1, file=csv_path))
print(f"wrote {display_paths(csv_path)} ({(OUT / 'table1.csv').stat().st_size} bytes)")
print()
print("\n".join((OUT / "table1.csv").read_text().splitlines()[:5]))

wrote ./out/table1.csv (1191 bytes)

,"Total","Nr"," 1"," 2"," 3","Os"," 4"," 5"," 6"," 7","Ws"," 8"," 9","10","Zd","11","12","99"
"Total",16847646.84,4373664.00,1986129.00,1809246.00,578289.00,3703896.00,124336.00,526279.00,2234995.00,818286.00,4576115.84,485326.00,3664559.84,426230.00,4193971.00,2752743.00,1441228.00,-
" 1",2711853.00,719059.00,398072.00,223990.00,96997.00,642258.00,36316.00,93589.00,345818.00,166535.00,648972.00,63767.00,537911.00,47294.00,701564.00,488628.00,212936.00,-
" 6",2320534.00,659680.00,348039.00,221332.00,90309.00,515003.00,32132.00,94957.00,251358.00,136556.00,543570.00,75442.00,430851.00,37277.00,602281.00,392395.00,209886.00,-
" 7",2505042.58,688962.00,354711.00,241913.00,92338.00,534147.00,25770.00,110930.00,251188.00,146259.00,663896.58,87305.00,515019.58,61572.00,618037.00,363490.00,254547.00,-


In [13]:
tab_path = OUT / "table1.tab"
eng.write_intermediate_table(0, str(tab_path), with_audit=True)
lines = tab_path.read_text().splitlines()
print(f"wrote {display_paths(tab_path)} ({tab_path.stat().st_size} bytes, {len(lines) - 1} rows)")
print()
print(f"header: {lines[0][:100]}…")
print(f"row 1:  {lines[1][:100]}…")

wrote ./out/table1.tab (11495 bytes, 125 rows)

header: "Total";"Total";16847646.84;42723;16847646.84;16847646.84;175677.00;141482.00;S;0.01;0.01;0.00;0.00;…
row 1:  "Total";"Nr";4373664.00;11395;4373664.00;4373664.00;43587.00;43293.00;S;0.01;0.01;0.00;0.00;0.00;0;0…


## Gotchas & known limitations

- **OPT/MOD time limits:** `opt_max_time` / `mod_max_time` are in
  minutes; `0` means no limit. Repeated fresh-engine runs with a
  one-minute limit pass; deadline expiry is not covered by that test.
- **GH / NET / CTA** require external executables and are not
  available in headless mode.
- **`audit()` mutates state** (it stores the realized bounds on
  the table); run it after your final suppression pass.
- The CLI mirrors every step above: `tauargus run`, `tables`,
  `suppress`, `round`, `audit`, `save`.

## Summary

You've now:

1. Protected a DataFrame with OPT/MOD and inspected masked responses
2. Computed batch tables and inspected the safety-rule results
3. Applied batch OPT suppression (and RND as an alternate flow)
4. Audited suppressed cells — realized feasibility intervals
5. Exported masked CSVs and retained INTERMEDIATE audit files

**Next:** [03_generators.ipynb](03_generators.ipynb) — generate
`.arb` / `.rda` / `.hrc` input files from Python.